Copyright Matlantis Corp. as contributors to Matlantis contrib project

# Steered MD (SMD) Simulation

Using PLUMED's `MOVINGRESTRAINT` feature, a Cu surface atom is gradually pulled in the z-direction to perform Steered MD.
This generates a trajectory that continuously explores the reaction coordinate space (z = 10.2 Å → 18.0 Å), which will be used to extract initial structures for umbrella sampling in the next step.

## Step 1: PLUMED Environment Setup
Steered MD uses PLUMED, an external library. The path and environment variables must be configured so that the kernel can be correctly called from Python.

**note**: For PLUMED installation instructions, see [00_README_en.ipynb](./00_README_en.ipynb). Modify the `PLUMED_ROOT` path according to your installation directory.

In [ ]:
# PLUMED environment

import os
import sys

# Path settings for PLUMED
PLUMED_ROOT   = os.path.expanduser("~/local/plumed-2.9.0")  # Modify to your PLUMED installation directory as needed
plumed_bin    = os.path.join(PLUMED_ROOT, "bin")
plumed_lib    = os.path.join(PLUMED_ROOT, "lib")
plumed_kernel = os.path.join(plumed_lib, "libplumedKernel.so")

# Set environment variables
os.environ["PATH"]            = f"{plumed_bin}:{os.environ.get('PATH', '')}"
os.environ["LD_LIBRARY_PATH"] = f"{plumed_lib}:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ["PLUMED_KERNEL"]   = str(plumed_kernel)

# Add Python library path
if str(PLUMED_ROOT) not in sys.path:
    sys.path.append(str(PLUMED_ROOT))

## Step 2: Import Libraries and Calculator Setup

Import the required ASE modules, PLUMED wrapper, and PFP Calculator, and configure the PFP calculation settings.

In [ ]:
import numpy as np

# ASE
from ase import units
from ase.io import read, write
from ase.constraints import FixAtoms
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.calculators.plumed import Plumed

# PFP
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator

from pfcc_extras import view_ngl

calc_mode = "PBE_PLUS_D3"
method_type = "PFVM_D3_PFVM"
model_version = "v8.0.0"
estimator = Estimator(calc_mode=calc_mode, method_type=method_type, model_version=model_version)
calculator = ASECalculator(estimator)

## Step 3: Load Initial Structure

Load the structure created by the NPzT equilibration MD and fix the bottom Cu slab layers (z < 4.0 Å).

**note**: If the file is not found in `output/02_md_equilibrium/`, it will be loaded from `assets/02_equilibrium_npzt_md/mdtraj_eq.xyz` to allow running this notebook independently.

In [ ]:
# Load from output, fall back to assets if not found
inp_file = './output/02_md_equilibrium/mdtraj_eq.xyz'
if not os.path.exists(inp_file):
    inp_file = './assets/02_equilibrium_npzt_md/mdtraj_eq.xyz'
    print(f"File not found in output, loading from assets: {inp_file}")

atoms = read(inp_file)
atoms.wrap()
v = view_ngl(atoms, representations="ball+stick")
display(v)

In [ ]:
# Fix the bottom 1st and 2nd Cu layers

thresh = 4.0
constraint = FixAtoms(mask=atoms.positions[:, 2] < thresh)
atoms.set_constraint(constraint)
constraint

## Step 4: Run Steered MD (PLUMED)

Use PLUMED's `MOVINGRESTRAINT` to gradually move the z-coordinate of atom 268 (Cu surface atom, ASE index: 267) from 10.2 Å to 18.0 Å.

**PLUMED configuration key points:**
* `UNITS LENGTH=A ENERGY=eV`: Set units to Å and eV.
* `POSITION ATOM=268`: Get the coordinates of the target atom (PLUMED uses **1-based** indexing).
* `MOVINGRESTRAINT`: Linearly move the harmonic potential center from `AT0` to `AT1`.

| Parameter | Value | Description |
|:---|:---|:---|
| Spring constant (KAPPA) | 10.0 eV/Å² | SMD restraint force |
| Start position (AT0) | 10.20 Å | Starting reaction coordinate value |
| End position (AT1) | 18.00 Å | Ending reaction coordinate value |
| Number of steps | 200,000 (= 200 ps) | |
| Temperature | 375 K | Langevin thermostat |
| Friction coefficient | 0.002 /fs | |

In [ ]:
atoms.get_positions()[267]

In [ ]:
out_dir = f"./output/03_steered_md_moving_restrain/"
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# PLUMED Settings　of Collective Variables
# ------------------------------------------------------------
cv_start = 10.2
cv_end   = 18.0
total_steps = 200_000

plumed_setting = [
    f"UNITS LENGTH=A ENERGY=eV",
    f"t: TIME",

    # Get the coordinates of the 268th Cu atom (PLUMED uses 1-based indexing)
    f"pos: POSITION ATOM=268",

    # SMD settings
    f"restraint: MOVINGRESTRAINT ARG=pos.z "
    f"STEP0=0             AT0={cv_start:.2f} KAPPA0=10.0 "
    f"STEP1={total_steps} AT1={cv_end:.2f}   KAPPA1=10.0",

    # Output settings
    f"PRINT STRIDE=100 ARG=pos.z,restraint.bias,restraint.force2,restraint.work FILE={out_dir}/COLVAR_SMD",
    f"FLUSH STRIDE=100",
]
print(plumed_setting)

# ------------------------------------------------------------
# Molecular Dynamics
# ------------------------------------------------------------
timestep = 1 * units.fs
ps = 1000 * units.fs
temperature = 375

# PLUMED
atoms.calc = Plumed(calc=calculator, input=plumed_setting, timestep=timestep, atoms=atoms, kT=1)

# Set the momenta corresponding to the given "temperature"
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature,force_temp=True)
Stationary(atoms)  # Set zero total momentum to avoid drifting

# Dynamics
dyn = Langevin(atoms, 
               timestep, 
               temperature_K=temperature, 
               friction=0.002/units.fs, 
               trajectory=f'{out_dir}/md-dyn.traj', 
               logfile=f'{out_dir}/md-dyn.log', 
               loginterval=100)

dyn.run(total_steps)

write(f'{out_dir}/md-smd-restart.xyz', atoms)

## Next Step
The Steered MD simulation is now complete.
In the next notebook [04_select_umbrella_sampling_initial_structures_en.ipynb](./04_select_umbrella_sampling_initial_structures_en.ipynb), the initial structures for each umbrella window will be extracted from this SMD trajectory.